In [2]:
import pytesseract
from PIL import Image
import cv2
import os
import re
import json

# Set the path to the Tesseract executable
tesseract_path = r'C:/Program Files/Tesseract-OCR/tesseract.exe'
pytesseract.pytesseract.tesseract_cmd = tesseract_path

def correct_image_orientation(image_path):
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    try:
        osd = pytesseract.image_to_osd(gray)
        angle = int(re.search(r'(?<=Rotate: )\d+', osd).group(0))

        if angle != 0:
            (h, w) = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, -angle, 1.0)
            rotated = cv2.warpAffine(image, M, (w, h))
        else:
            rotated = image

    except pytesseract.TesseractError as e:
        print(f"Tesseract OSD Error: {e}")
        rotated = image

    # Save the corrected image
    corrected_image_path = 'corrected_' + os.path.basename(image_path)
    cv2.imwrite(corrected_image_path, rotated)
    return corrected_image_path

def extract_text(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Correct the image orientation
    corrected_image_path = correct_image_orientation(image_path)
    
    image = Image.open(corrected_image_path)
    image = image.convert('RGB')
    image.save(corrected_image_path, dpi=(300, 300))
    
    # Perform OCR on the image
    try:
        text = pytesseract.image_to_string(image, lang='eng+nep')
    except pytesseract.TesseractError as e:
        print(f"Tesseract OCR Error: {e}")
        text = ""

    return text

def extract_specific_fields(text):
    fields = ["Full Name", "Date of Birth", "Sex", "Citizenship Certificate No"]
    extracted_info = {field: "" for field in fields}

    for line in text.split('\n'):
        for field in fields:
            if field.lower() in line.lower():
                extracted_info[field] = line.split(":")[-1].strip()
                break

    return extracted_info

def save_extracted_info(filename, extracted_info):
    folder_path = os.path.dirname(filename)
    os.makedirs(folder_path, exist_ok=True)

    with open(filename, 'w', encoding='utf-8') as file:
        json.dump(extracted_info, file, indent=4, ensure_ascii=False)

def main(image_path, output_file):
    text = extract_text(image_path)
    extracted_info = extract_specific_fields(text)

    for field, value in extracted_info.items():
        print(f"{field}: {value}")
    
    save_extracted_info(output_file, extracted_info)
    print(f"Extracted information saved to {output_file}")

if __name__ == "__main__":
    # Specify the local image path and output file
    image_path = 'X:/citizenship back.jpg'
    output_file = 'X:/extracted_info/extracted_info.json'
    
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    main(image_path, output_file)


Full Name: PRABIN TIWARI
Date of Birth: 16
Sex: Make
Citizenship Certificate No: 
Extracted information saved to X:/extracted_info/extracted_info.json
